In [1]:
from utils import (flag_rfi_channels, flag_outlier_dumps, vlsr_correction,
                    compute_cell_metrics, neighbor_qa, flag_outlier_pairs,
                    group_dumps_by_cell, compute_cell_gains,
                    apply_tsys_calibration, compute_R_for_dumps)
from plotters import plot_spectra_grid, plot_survey_mollweide
from ugradiolab import plotting
from ugradiolab.plotting import SS_MICRO, SS_FINE

from pathlib import Path
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages
import datetime as dt
import json
import numpy as np
import pandas as pd
import astropy.coordinates as ac
import astropy.units as u_ast
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- Hardware / signal-processing constants ---
SAMPLE_RATE_HZ = 3.2e6
NFFT = 1024
HI_REST_MHZ = 1420.405751768
C_KMS = 299792.458

F1_MHZ = 1419.86
F2_MHZ = 1421.14

T_CAL_00 = 58.0  # K, pol 0
T_CAL_11 = 79.0  # K, pol 1

HPBW_DEG = 3.4

# --- RFI channel flagger (utils.flag_rfi_channels) ---
RFI_WINDOW = 15
RFI_SIGMA = 10.0
RFI_CHEB_DEGREE = 3
RFI_SAMPLE_FRAC = 0.7
RFI_EXTREMA_ORDER = 2

# --- Outlier dump filter (utils.flag_outlier_dumps) ---
SHAPE_DEV_THRESH = 0.05
SHAPE_FRAC_THRESH = 0.05
SHAPE_MIN_GROUP_SIZE = 3

# --- Frequency switching (Section 4) ---
EDGE_TRIM_MHZ = 0.256

# --- Calibration (utils.compute_cell_gains / apply_tsys_calibration) ---
CAL_SPECTRUM_KEY = 'corr11'
CAL_GAIN_KEY = 'gain_11'

# --- Cross-session pair filter (utils.flag_outlier_pairs) ---
# Filter on T_B (K) so a session with anomalous T_sys becomes a population
# outlier in absolute units, not just spectral shape.
PAIR_NSIGMA = 5.0
PAIR_FRAC_THRESH = 0.15
MIN_VIABLE_PAIRS = 3
PAIR_SPECTRUM_KEY = 'T_B_lsr'

# --- Per-cell metrics (utils.compute_cell_metrics) ---
METRIC_MIN_VALID_CH = 8
METRIC_NOISE_V_MAX_KMS = -100.0
METRIC_SIGNAL_V_LO_KMS = -80.0
METRIC_SIGNAL_V_HI_KMS = 60.0
METRIC_SMOOTH_KERNEL = 5
METRIC_PEAK_MIN_SEP_KMS = 4.0
METRIC_PEAK_PROM_NSIGMA = 2.5
METRIC_MIN_NOISE_CH = 5

# --- Neighbor QA (utils.neighbor_qa) ---
# Operates on T_B (K), so W is in K*km/s. W_SCALE_FLOOR is the denominator
# floor for W_frac_resid: ~5 (R*km/s) * ~200 K T_sys ~ 1000 K*km/s.
# Brick-interleave grid (even b in {-4,-2,0,2,4}, odd b in {-3,-1,1,3} with
# l offset by Delta_l/2): 2.1 deg captures the 8 ring-1 neighbors -- 4 half-
# row diagonals at sep ~sqrt(2), and 4 same-axis neighbors at sep = 2.0.
NEIGHBOR_MAX_SEP_DEG = 2.1
MIN_NEIGHBORS = 2
W_Z_THRESH = 3.0
W_FRAC_THRESH = 0.30
W_SCALE_FLOOR = 1000.0
PEAK_V_Z_THRESH = 3.0
PEAK_V_ABS_THRESH = 15.0
PEAK_V_MIN_SIGMA = 3.0
PEAK_V_SCALE_FLOOR = 20.0
BIMODAL_MIN_RATIO = 0.68

# --- T_cal drift correction from periodic recal pointings (Section 5b) ---
# Use frequency-switched R(v_LSR) at fixed (RA, Dec) recal pointings as a
# gain- and T_cal-independent probe of T_sys(t), combined with the visit's
# own Y-factor to isolate T_cal drift. alpha(t) = T_cal_actual(t) / T_CAL_11
# is applied to every science cell's T_sys and T_B at its obs time.
# Set USE_REF_TCAL = False for the identity correction (no drift removal).
USE_REF_TCAL = True
REF_TCAL_ANCHOR_SESSION = 'main/session_001'
REF_TCAL_PATH = Path('artifacts/T_B_ref_circumpolar.npz')
REF_TCAL_REBUILD = False  # set True to force a rebuild from the anchor session
# LSR-velocity window for the R-shape projection. The recal pointings sit
# at b ~ +45 deg where HI is dominated by local emission near v_LSR ~ 0;
# tightening the window improves rho SNR by excluding line-free noise.
REF_TCAL_VEL_LO_KMS = -40.0
REF_TCAL_VEL_HI_KMS = 20.0
# Minimum LO1/LO2 noise-off pair count required for a visit to contribute
# alpha. Below this the rho estimator is dominated by noise.
REF_TCAL_MIN_PAIRS = 2
# Aggregate alpha by 'visit' or by 'session' median. 'session' is the
# safe default at this faint recal pointing -- per-visit rho is noise-
# limited at the few-percent level, so within-session timing is mostly
# noise rather than real T_cal variation.
REF_TCAL_AGGREGATE = 'session'
# Clamp alpha to this range; anything outside falls back to identity.
# A real diode drift beyond 2x indicates hardware failure, not calibration.
REF_TCAL_ALPHA_CLAMP = (0.5, 2.0)

# --- Paths / display ---
DATA_DIRS = [Path('data/main'), Path('data/nps')]
REOBSERVE_PATH = Path('artifacts/main_reobserve.json')
MOLL_CENTER_L = 120.0

%matplotlib inline

/home/ikaros/projects/ay-121/.venv/lib/python3.12/site-packages/rtlsdr/__init__.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. Load raw dumps

In [2]:
records = []
recal_records = []  # obs_recal_drift / cal_recal_drift (and _bk variants) -- handled separately
RECAL_CELL_PREFIXES = ('obs_recal_', 'cal_recal_')

for data_dir in DATA_DIRS:
    survey = data_dir.name  # 'main' or 'nps'
    for session_dir in sorted(data_dir.glob('session_*')):
        for cell_dir in sorted(session_dir.glob('obs_*')) + sorted(session_dir.glob('cal_*')):
            is_recal = cell_dir.name.startswith(RECAL_CELL_PREFIXES)
            for p in sorted(cell_dir.glob('*.npz')):
                with np.load(p, allow_pickle=True) as f:
                    rec = {
                        'path': p,
                        'survey': survey,
                        'session': f'{survey}/{session_dir.name}',
                        'target': str(f['target_name']),
                        'corr00': f['corr00'].astype(float),
                        'corr11': f['corr11'].astype(float),
                        'lo_mhz': float(f['lo_freq_mhz']),
                        'noise_on': bool(f['noise_on']),
                        'time': float(f['time']),
                        'alt': float(f['alt_deg']),
                        'az': float(f['az_deg']),
                        'ra': float(f['ra_deg']),
                        'dec': float(f['dec_deg']),
                    }
                if is_recal:
                    # Recal records are kept separate so they do NOT feed the
                    # per-cell calibration or pair-build steps. They are
                    # processed in a dedicated drift-correction cell below.
                    recal_records.append(rec)
                else:
                    records.append(rec)

N = len(records)
N_recal = len(recal_records)

# Galactic coordinates from RA/Dec
for r in records:
    c = ac.SkyCoord(ra=r['ra'] * u_ast.deg, dec=r['dec'] * u_ast.deg, frame='icrs')
    r['gl'] = round(c.galactic.l.deg, 2)
    r['gb'] = round(c.galactic.b.deg)

# Recal dumps share a single sky pointing per target, so (gl, gb) is constant
# per target. Compute it once (for diagnostics) but DO NOT use it as a cell key
# downstream -- visits are grouped by timestamp in the drift-correction cell.
for r in recal_records:
    c = ac.SkyCoord(ra=r['ra'] * u_ast.deg, dec=r['dec'] * u_ast.deg, frame='icrs')
    r['gl'] = round(c.galactic.l.deg, 2)
    r['gb'] = round(c.galactic.b.deg)

sessions = sorted(set(r['session'] for r in records))
lo_unique = sorted(set(r['lo_mhz'] for r in records if not r['noise_on']))
n_cal = sum(1 for r in records if r['noise_on'])
n_sci = sum(1 for r in records if not r['noise_on'])
n_recal_cal = sum(1 for r in recal_records if r['noise_on'])
n_recal_obs = sum(1 for r in recal_records if not r['noise_on'])

# Integration time per dump (seconds): NBLOCKS * NSAMPLES / fs.
# Main pipeline post-rewrite uses NSAMPLES=16384 (was 32768 pre-rewrite).
INT_TIME_S = 1025 * 16384 / SAMPLE_RATE_HZ  # ~5.25 s


def _utc(ts):
    return dt.datetime.fromtimestamp(ts, dt.UTC).strftime('%Y-%m-%d %H:%M')


session_data = []
for s in sessions:
    s_records = [r for r in records if r['session'] == s]
    n_dumps = len(s_records)
    if n_dumps == 0:
        continue
    n_cells = len(set((r['gl'], r['gb']) for r in s_records if not r['noise_on']))
    times = [r['time'] for r in s_records]
    t0, t1 = min(times), max(times)
    wall_s = t1 - t0
    duty_pct = (INT_TIME_S * n_dumps / wall_s) * 100 if wall_s > 0 else float('nan')
    n_recal_this = sum(1 for r in recal_records if r['session'] == s)
    session_data.append({
        'Session': s,
        'UTC Start': _utc(t0),
        'UTC End':   _utc(t1),
        'Wall (min)': round(wall_s / 60, 1),
        'Obs. Cells': n_cells,
        'Dumps': n_dumps,
        'Recal Dumps': n_recal_this,
        'Duty %': round(duty_pct, 1),
    })
display(pd.DataFrame(session_data))

print(f'Total: {N} science dumps ({n_sci} obs, {n_cal} cal), '
      f'{len(sessions)} sessions, LO: {lo_unique}')
print(f'Recal: {N_recal} dumps ({n_recal_obs} obs, {n_recal_cal} cal) -- handled separately')

,Session,UTC Start,UTC End,Wall (min),Obs. Cells,Dumps,Recal Dumps,Duty %
0,main/session_002,2026-05-14 01:56,2026-05-14 02:40,44.0,24,288,48,57.3


Total: 288 science dumps (192 obs, 96 cal), 1 sessions, LO: [1419.86, 1421.14]
Recal: 96 dumps (64 obs, 32 cal) -- handled separately


## 1b. Sun + Moon proximity check

Verify every observed dump is at least `SUN_AVOID_DEG` / `MOON_AVOID_DEG` from the Sun and Moon at its time of capture (the planner enforces this prospectively, but a long-running session can drift if either body has crossed onto a cell since planning).


In [3]:
from astropy.coordinates import get_sun, get_body
from astropy.time import Time

SUN_AVOID_DEG = 30.0
MOON_AVOID_DEG = 10.0

ts = np.array([r['time'] for r in records])
ras = np.deg2rad([r['ra'] for r in records])
decs = np.deg2rad([r['dec'] for r in records])

# Sun moves ~0.04 deg/min, Moon ~0.5 deg/h.  A 10-min grid + linear interp
# of (cos ra, sin ra, dec) is well below either threshold (Sun=30 deg, Moon=10 deg).
GRID_SEC = 600.0
t_lo, t_hi = ts.min() - 1, ts.max() + 1
t_grid = np.arange(t_lo, t_hi + GRID_SEC, GRID_SEC)
atime = Time(t_grid, format='unix')

def _interp_radec(body):
    ra = np.deg2rad(body.ra.deg)
    dec = np.deg2rad(body.dec.deg)
    ra_x = np.interp(ts, t_grid, np.cos(ra))
    ra_y = np.interp(ts, t_grid, np.sin(ra))
    return np.arctan2(ra_y, ra_x), np.interp(ts, t_grid, dec)

sun_ra_t,  sun_dec_t  = _interp_radec(get_sun(atime))
moon_ra_t, moon_dec_t = _interp_radec(get_body('moon', atime))

def _sep_deg(ra_b, dec_b):
    cos_sep = (np.sin(decs) * np.sin(dec_b)
               + np.cos(decs) * np.cos(dec_b) * np.cos(ras - ra_b))
    return np.rad2deg(np.arccos(np.clip(cos_sep, -1.0, 1.0)))

sun_sep  = _sep_deg(sun_ra_t,  sun_dec_t)
moon_sep = _sep_deg(moon_ra_t, moon_dec_t)

for body, sep, thr in [('Sun', sun_sep, SUN_AVOID_DEG),
                       ('Moon', moon_sep, MOON_AVOID_DEG)]:
    bad = sep < thr
    n_bad = int(bad.sum())
    print(f'{body:4s} proximity: {len(records)} dumps, threshold {thr:.0f} deg, '
          f'min={sep.min():.1f} deg, median={np.median(sep):.1f} deg')
    if n_bad:
        print(f'  WARNING: {n_bad} dumps within {thr:.0f} deg of the {body}')
        for i in np.argsort(sep)[:min(10, n_bad)]:
            r = records[i]
            print(f"    sep={sep[i]:5.2f} deg  l={r['gl']:6.2f} b={r['gb']:+3d}  "
                  f"{r['session']}/{Path(r['path']).name}")
    else:
        print(f'  All clear.')


Sun  proximity: 288 dumps, threshold 30 deg, min=77.1 deg, median=81.4 deg
  All clear.
Moon proximity: 288 dumps, threshold 10 deg, min=106.9 deg, median=110.2 deg
  All clear.


## 2. fftshift + RFI flagging + outlier dump filter

In [4]:
DC_BIN = NFFT // 2
f_bb_mhz = np.fft.fftshift(np.fft.fftfreq(NFFT, d=1.0 / SAMPLE_RATE_HZ)) / 1e6
df_khz = SAMPLE_RATE_HZ / NFFT / 1e3

# Apply fftshift + RFI flagging uniformly to science AND recal records so the
# drift-correction step sees the same baseband / RFI treatment as the cell
# calibration step.
for r in records + recal_records:
    r['corr00'] = np.fft.fftshift(r['corr00'])
    r['corr11'] = np.fft.fftshift(r['corr11'])
    r['stokes_I'] = r['corr11']  # pol 1 only -- pol 0 has bad noise-diode coupling at 3.2 MHz

n_flagged = sum(
    flag_rfi_channels(
        r['stokes_I'],
        window=RFI_WINDOW,
        sigma_thresh=RFI_SIGMA,
        degree=RFI_CHEB_DEGREE,
        sample_frac=RFI_SAMPLE_FRAC,
        extrema_order=RFI_EXTREMA_ORDER,
    )
    for r in records + recal_records
)

print(f'Baseband: [{f_bb_mhz[0]:.3f}, {f_bb_mhz[-1]:.3f}] MHz, '
      f'dnu = {df_khz:.2f} kHz')
print(f'RFI flagged: {n_flagged} samples across {N + N_recal} dumps '
      f'({N} science + {N_recal} recal)')
print('NOTE: using pol 1 (corr11) only')

Baseband: [-1.600, 1.597] MHz, dnu = 3.12 kHz
RFI flagged: 1355 samples across 384 dumps (288 science + 96 recal)
NOTE: using pol 1 (corr11) only


In [5]:
outlier_records = flag_outlier_dumps(
    records,
    dev_thresh=SHAPE_DEV_THRESH,
    frac_thresh=SHAPE_FRAC_THRESH,
    min_group_size=SHAPE_MIN_GROUP_SIZE,
)
N = len(records)

print(f'Outlier dump filter (spectral shape): removed {len(outlier_records)} dumps')
print(f'  (flag if >{SHAPE_FRAC_THRESH:.0%} of channels deviate '
      f'>{SHAPE_DEV_THRESH:.0%} from group median)')
print(f'Remaining: {N} dumps')

Outlier dump filter (spectral shape): removed 0 dumps
  (flag if >5% of channels deviate >5% from group median)
Remaining: 288 dumps


## 4. Frequency switching (R)

Pair dumps at f1 and f2, compute R = (I_f1 - I_f2) / I_f2.

In [6]:
lo1, lo2 = F1_MHZ, F2_MHZ
f_sky = lo1 + f_bb_mhz
f_sky_2 = lo2 + f_bb_mhz
f_overlap_lo = max(f_sky[0], f_sky_2[0]) + EDGE_TRIM_MHZ
f_overlap_hi = min(f_sky[-1], f_sky_2[-1]) - EDGE_TRIM_MHZ
overlap_mask = (f_sky >= f_overlap_lo) & (f_sky <= f_overlap_hi)
f_overlap = f_sky[overlap_mask]
v_overlap = C_KMS * (1 - f_overlap / HI_REST_MHZ)
dv_kms = np.abs(np.median(np.diff(v_overlap)))

print(f'LO pair: ({lo1}, {lo2}) MHz')
print(f'Overlap: [{f_overlap_lo:.2f}, {f_overlap_hi:.2f}] MHz')
print(f'Velocity (topo): [{v_overlap[-1]:.0f}, {v_overlap[0]:.0f}] km/s')
print(f'dv = {dv_kms:.3f} km/s per channel')

# Frequency-switched R per (session, cell)
all_pointings = sorted(set((r['gl'], r['gb']) for r in records
                           if r['gl'] is not None))

cell_results = {}
for gl, gb in all_pointings:
    sci_dumps = [r for r in records
                 if r['gl'] == gl and r['gb'] == gb and not r['noise_on']]
    for dr in sessions:
        dr_dumps = [r for r in sci_dumps if r['session'] == dr]
        result = compute_R_for_dumps(dr_dumps, lo1, lo2, overlap_mask, v_overlap)
        if result is not None:
            cell_results[(dr, gl, gb)] = result

LO pair: (1419.86, 1421.14) MHz
Overlap: [1419.80, 1421.20] MHz
Velocity (topo): [-168, 128] km/s
dv = 0.660 km/s per channel


/home/ikaros/projects/ay-121/labs/04/utils/freqswitch.py:107: RuntimeWarning: Mean of empty slice
  R_mean = np.nanmean(R_cat, axis=0)


## 5. Per-cell gain calibration

Each cell has its own cal-on/cal-off pair. Compute gain per cell,
then T_sys = P_off / gain. Convert R -> T_B = R * T_sys.

In [7]:
dc_mask = np.ones(NFFT, dtype=bool)
dc_mask[DC_BIN] = False

# Per-cell channel-dependent gain g(nu) and T_sys(nu) from pol 1 only
# (pol 0 has unreliable noise-diode coupling at 3.2 MHz). The smoothed gain
# spectrum is inverted to T_sys(nu); the scalar 'T_sys' summary is the band
# median over the overlap region.
cal_dumps, obs_dumps_by_cell = group_dumps_by_cell(records)
cell_gains = compute_cell_gains(
    cal_dumps, obs_dumps_by_cell, dc_mask, T_CAL_11,
    spectrum_key=CAL_SPECTRUM_KEY, gain_key=CAL_GAIN_KEY,
)
print(f'Gain computed for {len(cell_gains)} (session, cell) entries')

cell_results_TB = apply_tsys_calibration(
    cell_results, cell_gains, obs_dumps_by_cell, dc_mask,
    spectrum_key=CAL_SPECTRUM_KEY, gain_key=CAL_GAIN_KEY,
    overlap_mask=overlap_mask,
)
print(f'Calibrated {len(cell_results_TB)} (session, cell) entries')

CAL_GAIN_SCALAR_KEY = CAL_GAIN_KEY + '_scalar'

# Gain summary (band-median scalars; full T_sys(nu) lives on each entry).
gain_rows = []
for (dr, gl, gb), v in sorted(cell_results_TB.items()):
    gain_rows.append({
        'session': dr, 'gl': gl, 'gb': gb,
        CAL_GAIN_KEY: v[CAL_GAIN_SCALAR_KEY],
        'T_sys': v['T_sys'],
    })
gain_df = pd.DataFrame(gain_rows)
if len(gain_df):
    print(f'\nGain stats (pol 1 only, band-median scalars):')
    print(f'  {CAL_GAIN_KEY}: {gain_df[CAL_GAIN_KEY].median():.2f} (median), '
          f'range [{gain_df[CAL_GAIN_KEY].min():.2f}, {gain_df[CAL_GAIN_KEY].max():.2f}]')
    print(f'  T_sys:   {gain_df["T_sys"].median():.0f} K (median), '
          f'range [{gain_df["T_sys"].min():.0f}, {gain_df["T_sys"].max():.0f}] K')

    # Spectral T_sys diagnostics: how flat is T_sys(nu) within each cell?
    tsys_overlap_arrs = [v['T_sys_overlap']
                         for v in cell_results_TB.values()
                         if np.isfinite(v['T_sys'])]
    if tsys_overlap_arrs:
        rel_spread = np.array([
            float(np.nanstd(a) / np.nanmedian(a))
            for a in tsys_overlap_arrs
            if np.isfinite(np.nanmedian(a)) and np.nanmedian(a) > 0
        ])
        if rel_spread.size:
            print(f'  T_sys(nu) relative spread within overlap: '
                  f'median {np.median(rel_spread)*100:.1f}%, '
                  f'p95 {np.percentile(rel_spread, 95)*100:.1f}%')

Gain computed for 24 (session, cell) entries
Calibrated 24 (session, cell) entries

Gain stats (pol 1 only, band-median scalars):
  gain_11: 2.64 (median), range [2.31, 2.95]
  T_sys:   231 K (median), range [215, 256] K
  T_sys(nu) relative spread within overlap: median 8.7%, p95 9.0%


/home/ikaros/projects/ay-121/labs/04/utils/calibration.py:165: RuntimeWarning: Mean of empty slice
  P_on = np.nanmean([c[spectrum_key] for c in cals], axis=0)
/home/ikaros/projects/ay-121/labs/04/utils/calibration.py:166: RuntimeWarning: Mean of empty slice
  P_off = np.nanmean([o[spectrum_key] for o in obs], axis=0)
/home/ikaros/projects/ay-121/labs/04/utils/calibration.py:219: RuntimeWarning: Mean of empty slice
  P_off_nu = np.nanmean([o[spectrum_key] for o in obs], axis=0)


## 5b. T_cal drift correction from periodic recal pointings

Each session inserts a fixed (RA, Dec) recal pointing (`recal_drift` at
RA=180, Dec=72, plus the backup `recal_drift_bk` at RA=90, Dec=72) every
`RECAL_EVERY_N_CELLS` survey cells. Both targets are circumpolar from
Leuschner, so the *sky* contribution at the pointing is constant in
LSR-velocity space across all visits and all sessions. Any time
variation in the calibration must therefore be instrumental.

For each recal visit we measure two gain- and time-stable observables
(the only two needed to isolate diode drift):

```
Y(t)            = median over band of P_on / P_off
                = 1 + T_cal(t) / T_sys_recal(t)
R(t, v_LSR)     = (I_LO1 - I_LO2) / I_LO2
                = Delta_T_B_sky(v_LSR) / T_sys_recal(t)
```

`R(t, v)` is gain- AND T_cal-independent. Its spectral shape
`Delta_T_B_sky(v_LSR)` is genuinely constant across visits once we
resample R onto a common LSR-velocity grid (a fixed *topocentric*
channel maps to different LSR velocities at different times, so any
cross-visit averaging done in topocentric space smears the line).

A single scalar amplitude per visit captures everything we need:

```
rho(t) = sum_v R_shape(v) * R(t, v)  /  sum_v R_shape(v)^2
alpha(t) = (Y(t) - 1) / (Y_anchor - 1)  *  (rho_anchor / rho(t))
         = T_cal_actual(t) / T_CAL_11
```

Where `R_shape(v_LSR)` is the simple mean of the anchor-session
recal-visit R-spectra, `Y_anchor` is the anchor session's median Y,
and `rho_anchor` is the mean projection of anchor visits onto
`R_shape` (=1 by construction; carried for diagnostics). By
construction the anchor session's mean alpha equals 1: alpha at any
other time measures the *relative* diode drift versus the anchor.

Applying `alpha(t_cell)` to a science cell scales its `T_sys` and
`T_B` by exactly the diode-drift factor: the cell's own Y-factor
and elevation-dependent T_sys are already baked into its pipeline
calibration, and `alpha` is purely a multiplicative T_cal correction.

`R_shape` and `Y_anchor` are computed once from the anchor session
and cached to `REF_TCAL_PATH`. The cache is invalidated automatically
if the LSR grid or `REF_TCAL_ANCHOR_SESSION` changes; set
`REF_TCAL_REBUILD = True` to force a rebuild.

Diagnostics printed below:

- `alpha(t)` distribution -- should hover near 1 with ~5-15% scatter.
- `alpha` vs alt(recal) -- slope and Pearson r should be near 0;
  a clear slope indicates an elevation-dependent leak (atmosphere,
  ground spillover, RFI, or an LSR-regridding bug).
- Cross-target check (`recal_drift` vs `recal_drift_bk`) at matched
  times -- relative spread should match the per-visit noise.

If `USE_REF_TCAL = False`, or the anchor session has no R-paired recal
visits, alpha collapses to identity and no correction is applied.

In [8]:
# --- Group recal records into visits (timestamp clusters per session/target) ---

RECAL_VISIT_GAP_SEC = 300.0  # >5 min gap between dumps -> new visit


def _recal_target_id(rec):
    t = rec.get('target', '')
    if t.startswith('cal_') or t.startswith('obs_'):
        return t[4:]
    return t


recal_by_st = defaultdict(list)
for r in recal_records:
    recal_by_st[(r['session'], _recal_target_id(r))].append(r)


# --- Common LSR-velocity grid for all recal R(v) ---
# Every cross-visit / cross-session combination of R(v) MUST happen in
# LSR space: a fixed topocentric channel maps to different LSR velocities
# at different times, so averaging R(v_topo) across visits smears the
# sky line. We anchor the grid on the mean recal vcorr; per-visit R is
# interpolated onto this grid using each visit's own vcorr.
_recal_vcorr_list = [
    vlsr_correction(r['ra'], r['dec'], r['time']) for r in recal_records
]
RECAL_MEAN_VCORR = float(np.mean(_recal_vcorr_list)) if _recal_vcorr_list else 0.0
v_overlap_inc = v_overlap[::-1]
v_lsr_recal_inc = v_overlap_inc + RECAL_MEAN_VCORR

shape_mask_inc = (
    (v_lsr_recal_inc >= REF_TCAL_VEL_LO_KMS) &
    (v_lsr_recal_inc <= REF_TCAL_VEL_HI_KMS)
)


def _calibrate_visit(visit_dumps):
    """Per-visit reduction.

    Returns scalar Y_scalar, scalar T_sys_pipeline (= T_CAL_11/(Y-1)
    band-median; informational), and R(v) interpolated onto the common
    v_lsr_recal_inc grid. R_lsr_inc is None if the visit has no paired
    LO1/LO2 noise-off sample.
    """
    cals = [d for d in visit_dumps if d['noise_on']]
    obss = [d for d in visit_dumps if not d['noise_on']]
    if not cals or not obss:
        return None

    P_on = np.nanmean([c['corr11'] for c in cals], axis=0)
    P_off = np.nanmean([o['corr11'] for o in obss], axis=0)
    diff = P_on - P_off
    diff_safe = diff.astype(float).copy()
    diff_safe[diff_safe <= 0] = np.nan
    gain_raw = diff_safe / T_CAL_11
    P_off_masked = P_off.astype(float).copy()
    P_off_masked[~dc_mask] = np.nan
    T_sys_nu = P_off_masked / gain_raw
    T_sys_overlap = T_sys_nu[overlap_mask]
    diff_overlap = diff[overlap_mask]
    T_sys_pipeline = float(np.nanmedian(T_sys_overlap))
    g_T_cal_scalar = float(np.nanmedian(diff_overlap))
    with np.errstate(divide='ignore', invalid='ignore'):
        Y_nu = P_on / P_off
    Y_scalar = float(np.nanmedian(Y_nu[overlap_mask]))

    R_lsr_inc = None
    o1 = [d for d in obss if d['lo_mhz'] == F1_MHZ]
    o2 = [d for d in obss if d['lo_mhz'] == F2_MHZ]
    n_p = min(len(o1), len(o2))
    if n_p >= 1:
        I1 = np.nanmean([o['stokes_I'] for o in o1[:n_p]], axis=0)
        I2 = np.nanmean([o['stokes_I'] for o in o2[:n_p]], axis=0)
        with np.errstate(divide='ignore', invalid='ignore'):
            R = (I1 - I2) / I2
        R_overlap_inc = R[overlap_mask][::-1]
        t_mean = float(np.mean([d['time'] for d in visit_dumps]))
        ra0 = visit_dumps[0]['ra']
        dec0 = visit_dumps[0]['dec']
        v_corr = vlsr_correction(ra0, dec0, t_mean)
        v_visit_lsr_inc = v_overlap_inc + v_corr
        R_lsr_inc = np.interp(
            v_lsr_recal_inc, v_visit_lsr_inc, R_overlap_inc,
            left=np.nan, right=np.nan,
        )

    if not np.isfinite(T_sys_pipeline) or not np.isfinite(g_T_cal_scalar):
        return None

    return {
        't_mid': float(np.mean([d['time'] for d in visit_dumps])),
        'alt_mean': float(np.mean([d['alt'] for d in visit_dumps])),
        'Y_scalar': Y_scalar,
        'T_sys_pipeline': T_sys_pipeline,
        'g_T_cal': g_T_cal_scalar,
        'R_lsr_inc': R_lsr_inc,
        'n_cal': len(cals),
        'n_obs': len(obss),
        'n_pairs': n_p,
    }


recal_visits = defaultdict(list)
for key, dumps in recal_by_st.items():
    dumps.sort(key=lambda r: r['time'])
    if not dumps:
        continue
    current = [dumps[0]]
    for r in dumps[1:]:
        if r['time'] - current[-1]['time'] > RECAL_VISIT_GAP_SEC:
            v = _calibrate_visit(current)
            if v is not None:
                recal_visits[key].append(v)
            current = [r]
        else:
            current.append(r)
    v = _calibrate_visit(current)
    if v is not None:
        recal_visits[key].append(v)

n_visits_total = sum(len(v) for v in recal_visits.values())
print(f'Recal visits: {n_visits_total} across {len(recal_visits)} (session, target) pairs')
for key, vs in sorted(recal_visits.items()):
    if not vs:
        continue
    T_sys_arr = np.array([v['T_sys_pipeline'] for v in vs])
    Y_arr = np.array([v['Y_scalar'] for v in vs])
    alt_arr = np.array([v['alt_mean'] for v in vs])
    n_with_R = sum(1 for v in vs if v['R_lsr_inc'] is not None
                   and v['n_pairs'] >= REF_TCAL_MIN_PAIRS)
    print(f'  {key[0]} / {key[1]}: {len(vs)} visits ({n_with_R} with R, n_pairs>={REF_TCAL_MIN_PAIRS}), '
          f'T_sys_pipe={np.median(T_sys_arr):.1f} K, '
          f'Y={np.median(Y_arr):.4f}, alt={np.median(alt_arr):.1f} deg')


# --- Build R_shape(v_LSR) and Y_anchor per target from the anchor session ---

R_shape_by_target = {}
ref_target_meta = {}


def _project(R_shape, R_visit):
    """rho = <R_shape, R_visit> / <R_shape, R_shape> on the shape support."""
    sel = (shape_mask_inc & np.isfinite(R_shape) & np.isfinite(R_visit))
    if int(sel.sum()) < 4:
        return float('nan')
    num = float(np.nansum(R_shape[sel] * R_visit[sel]))
    den = float(np.nansum(R_shape[sel] ** 2))
    if den <= 0 or not np.isfinite(den):
        return float('nan')
    return num / den


def _build_R_shape_from_anchor():
    out_shape = {}
    out_meta = {}
    for (sess, tgt), vs in recal_visits.items():
        if sess != REF_TCAL_ANCHOR_SESSION:
            continue
        vs_ok = [v for v in vs if v['R_lsr_inc'] is not None
                 and v['n_pairs'] >= REF_TCAL_MIN_PAIRS]
        if not vs_ok:
            continue
        R_stack = np.array([v['R_lsr_inc'] for v in vs_ok])
        R_shape = np.nanmean(R_stack, axis=0)
        Y_arr = np.array([v['Y_scalar'] for v in vs_ok
                          if np.isfinite(v['Y_scalar'])])
        if Y_arr.size == 0 or not np.isfinite(np.median(Y_arr)):
            continue
        Y_anchor = float(np.median(Y_arr))
        rhos = [_project(R_shape, v['R_lsr_inc']) for v in vs_ok]
        rhos = [r for r in rhos if np.isfinite(r)]
        rho_anchor = float(np.mean(rhos)) if rhos else 1.0
        out_shape[tgt] = R_shape
        out_meta[tgt] = {
            'n_visits': len(vs_ok),
            't_min': float(min(v['t_mid'] for v in vs_ok)),
            't_max': float(max(v['t_mid'] for v in vs_ok)),
            'Y_anchor': Y_anchor,
            'rho_anchor': rho_anchor,
            'anchor_session': REF_TCAL_ANCHOR_SESSION,
        }
    return out_shape, out_meta


cache_was_loaded = False

if not USE_REF_TCAL:
    print('USE_REF_TCAL = False -- using identity alpha.')
elif REF_TCAL_PATH.exists() and not REF_TCAL_REBUILD:
    cached_shape = {}
    cached_meta = {}
    cached_v_lsr = None
    with np.load(REF_TCAL_PATH, allow_pickle=True) as f:
        for k in f.files:
            if k.startswith('shape_'):
                cached_shape[k[len('shape_'):]] = f[k]
            elif k == 'v_lsr_recal_inc':
                cached_v_lsr = f[k]
            elif k == 'meta':
                cached_meta = f[k].item()
    grid_ok = (cached_v_lsr is not None
               and cached_v_lsr.shape == v_lsr_recal_inc.shape
               and np.allclose(cached_v_lsr, v_lsr_recal_inc))
    cfg_ok = bool(cached_meta) and all(
        m.get('anchor_session') == REF_TCAL_ANCHOR_SESSION
        for m in cached_meta.values()
    )
    if not grid_ok or not cfg_ok:
        reason = 'v_lsr_recal grid mismatch' if not grid_ok else \
                 f'anchor != {REF_TCAL_ANCHOR_SESSION!r}'
        print(f'Cache invalid ({reason}) -- rebuilding R_shape from anchor.')
        R_shape_by_target, ref_target_meta = _build_R_shape_from_anchor()
    else:
        R_shape_by_target = {k: np.asarray(v) for k, v in cached_shape.items()}
        ref_target_meta = cached_meta
        cache_was_loaded = True
        print(f'Loaded R_shape from {REF_TCAL_PATH} '
              f'({len(R_shape_by_target)} target(s): {sorted(R_shape_by_target)}, '
              f'anchor: {REF_TCAL_ANCHOR_SESSION})')
else:
    R_shape_by_target, ref_target_meta = _build_R_shape_from_anchor()

if USE_REF_TCAL and R_shape_by_target and not cache_was_loaded:
    out = {f'shape_{tgt}': arr for tgt, arr in R_shape_by_target.items()}
    out['v_lsr_recal_inc'] = v_lsr_recal_inc
    out['meta'] = np.array(ref_target_meta, dtype=object)
    REF_TCAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(REF_TCAL_PATH, **out)
    print(f'Wrote {REF_TCAL_PATH} '
          f'({len(R_shape_by_target)} target(s), anchor: {REF_TCAL_ANCHOR_SESSION}).')


# --- Per-visit alpha: (Y(t)-1) / (Y_anchor-1) * (rho_anchor / rho(t)) ---

for (sess, tgt), vs in recal_visits.items():
    R_shape = R_shape_by_target.get(tgt) if USE_REF_TCAL else None
    meta = ref_target_meta.get(tgt) if USE_REF_TCAL else None
    Y_anchor = meta['Y_anchor'] if meta else None
    rho_anchor = meta.get('rho_anchor', 1.0) if meta else None
    for v in vs:
        alpha = float('nan')
        rho = float('nan')
        if (R_shape is not None and Y_anchor is not None
                and v['R_lsr_inc'] is not None
                and v['n_pairs'] >= REF_TCAL_MIN_PAIRS
                and np.isfinite(v['Y_scalar'])
                and abs(Y_anchor - 1.0) > 1e-6):
            rho = _project(R_shape, v['R_lsr_inc'])
            if np.isfinite(rho) and rho > 0:
                alpha = ((v['Y_scalar'] - 1.0) / (Y_anchor - 1.0)
                         * (rho_anchor / rho))
        v['rho'] = rho
        v['alpha_target'] = alpha


# --- Aggregate alpha to either per-visit points or per-session medians ---
# Per-visit is mathematically cleanest but rho is noise-dominated at the
# faint recal pointing, so the default is per-session: one alpha per
# session, anchored at the median visit time. This averages out per-visit
# noise while preserving cross-session drift (the thing we actually want).

ALPHA_LO, ALPHA_HI = REF_TCAL_ALPHA_CLAMP

if REF_TCAL_AGGREGATE == 'session':
    alpha_points_raw = []
    alpha_session_summary = {}  # session -> (t_mid, alpha, n_visits, [targets])
    for sess in sessions:
        alphas = []
        ts = []
        tgts_used = []
        for (s, tgt), vs in recal_visits.items():
            if s != sess:
                continue
            sess_alphas = [v['alpha_target'] for v in vs
                           if np.isfinite(v.get('alpha_target', np.nan))]
            if sess_alphas:
                tgts_used.append(tgt)
                alphas.extend(sess_alphas)
                ts.extend(v['t_mid'] for v in vs
                          if np.isfinite(v.get('alpha_target', np.nan)))
        if alphas:
            a = float(np.median(alphas))
            t = float(np.median(ts))
            alpha_points_raw.append((t, a))
            alpha_session_summary[sess] = (t, a, len(alphas), tgts_used)
else:
    alpha_points_raw = []
    alpha_session_summary = {}
    for (sess, tgt), vs in recal_visits.items():
        for v in vs:
            a = v.get('alpha_target')
            if np.isfinite(a):
                alpha_points_raw.append((v['t_mid'], float(a)))

# Clamp + drop outliers
alpha_points = []
n_clamped = 0
for t, a in alpha_points_raw:
    if not np.isfinite(a):
        continue
    if a < ALPHA_LO or a > ALPHA_HI:
        n_clamped += 1
        continue
    alpha_points.append((t, a))


# --- Diagnostics ---

if alpha_points:
    arr = np.array([a for _, a in alpha_points])
    print(f'\nalpha points ({REF_TCAL_AGGREGATE} aggregation): '
          f'{len(alpha_points)} kept, {n_clamped} dropped outside '
          f'[{ALPHA_LO}, {ALPHA_HI}]')
    print(f'  alpha distribution: median={np.median(arr):.4f}, '
          f'p10={np.percentile(arr, 10):.4f}, p90={np.percentile(arr, 90):.4f}, '
          f'range=[{arr.min():.4f}, {arr.max():.4f}]')
else:
    print(f'\nNo finite alpha points after clamp '
          f'({n_clamped} dropped) -- identity correction will be applied.')

# Per-visit diagnostics: alpha vs alt, target cross-check.
visit_alt_alpha = []
target_alphas_at_time = defaultdict(list)
for (sess, tgt), vs in recal_visits.items():
    for v in vs:
        a = v.get('alpha_target')
        if np.isfinite(a):
            visit_alt_alpha.append((v['alt_mean'], float(a), v['t_mid'], tgt))
            target_alphas_at_time[(sess, round(v['t_mid'] / RECAL_VISIT_GAP_SEC))].append(
                (tgt, float(a)))

if len(visit_alt_alpha) >= 4:
    alts = np.array([x[0] for x in visit_alt_alpha])
    alphs = np.array([x[1] for x in visit_alt_alpha])
    a = alts - alts.mean()
    b = alphs - alphs.mean()
    denom = float(np.sqrt(np.sum(a * a) * np.sum(b * b)))
    r = float(np.sum(a * b) / denom) if denom > 0 else float('nan')
    slope = float(np.polyfit(alts, alphs, 1)[0])
    print(f'  alpha vs alt (per visit, all values): slope={slope:+.4f}/deg, '
          f'Pearson r={r:+.3f} (target: ~0)')

# Cross-target check at matched times
cross = []
for key, items in target_alphas_at_time.items():
    tgts = {t for t, _ in items}
    if len(tgts) >= 2:
        meds = {t: np.median([a for tt, a in items if tt == t]) for t in tgts}
        spread = max(meds.values()) - min(meds.values())
        cross.append(spread / max(np.mean(list(meds.values())), 1e-9))
if cross:
    print(f'  cross-target spread: {len(cross)} time-matched groups, '
          f'median |Delta alpha|/alpha = {np.median(cross):.3f}, '
          f'max = {max(cross):.3f}')


# --- Global alpha(t) interpolator (piecewise-linear, constant extrapolation) ---

def _make_alpha_interp(points):
    pts = sorted((t, a) for t, a in points
                 if np.isfinite(t) and np.isfinite(a) and a > 0)
    if len(pts) < 2:
        if len(pts) == 1:
            a0 = pts[0][1]
            return lambda t, _a=a0: float(_a)
        return lambda t: 1.0
    ts = np.array([p[0] for p in pts])
    alphas = np.array([p[1] for p in pts])

    def _interp(t):
        return float(np.interp(t, ts, alphas, left=alphas[0], right=alphas[-1]))

    return _interp


alpha_global = _make_alpha_interp(alpha_points)


# --- Mean obs timestamp per (session, gl, gb) for alpha lookup ---

obs_time_by_sc = {}
for key, obs_list in obs_dumps_by_cell.items():
    times = [r['time'] for r in obs_list if 'time' in r]
    if times:
        obs_time_by_sc[key] = float(np.mean(times))


# --- Apply alpha to cell_results_TB in place ---

ALPHA_KEYS_TO_SCALE = ('T_B_overlap', 'T_sys_nu', 'T_sys_overlap')
alpha_applied = {}
n_applied = 0
n_identity = 0
for key, entry in cell_results_TB.items():
    t_obs = obs_time_by_sc.get(key)
    alpha = float(alpha_global(t_obs)) if t_obs is not None else 1.0
    if not np.isfinite(alpha) or alpha <= 0:
        alpha = 1.0
    # Defensive clamp on the interpolated value too
    if alpha < ALPHA_LO or alpha > ALPHA_HI:
        alpha = 1.0
    alpha_applied[key] = alpha
    entry['drift_alpha'] = alpha
    if alpha == 1.0:
        n_identity += 1
        continue
    for k in ALPHA_KEYS_TO_SCALE:
        if k in entry and entry[k] is not None:
            entry[k] = entry[k] * alpha
    if np.isfinite(entry.get('T_sys', np.nan)):
        entry['T_sys'] = entry['T_sys'] * alpha
    n_applied += 1


# --- Per-session diagnostics + populate visit fields consumed downstream ---

T_sys_true_by_session = {}
T_cal_true_by_session = {}
recal_session_summary = []
for sess in sessions:
    alphas_sess = []
    tsys_pipe_sess = []
    targets_used = []
    for (s, tgt), vs in recal_visits.items():
        if s != sess or not vs:
            continue
        targets_used.append(tgt)
        for v in vs:
            a = float(v.get('alpha_target', float('nan')))
            v['alpha_abs'] = a
            v['T_sys_true'] = float(v['T_sys_pipeline'] * a) if np.isfinite(a) else float('nan')
            v['T_cal_true'] = float(T_CAL_11 * a) if np.isfinite(a) else float('nan')
            if np.isfinite(a):
                alphas_sess.append(a)
                tsys_pipe_sess.append(v['T_sys_pipeline'])
    if alphas_sess:
        med_a = float(np.median(alphas_sess))
        T_sys_true_by_session[sess] = float(np.median(
            np.array(alphas_sess) * np.array(tsys_pipe_sess)))
        T_cal_true_by_session[sess] = T_CAL_11 * med_a
    recal_session_summary.append({
        'session': sess,
        'n_alpha_points': len(alphas_sess),
        'targets': targets_used,
    })


# Per-session APPLIED alpha (post-clamp, interpolated from the kept points).
# This is what each session's cells were actually scaled by.
applied_alpha_by_session = defaultdict(list)
for key, a in alpha_applied.items():
    applied_alpha_by_session[key[0]].append(a)

print(f'\nDrift correction applied: {n_applied} cells scaled, '
      f'{n_identity} unchanged (alpha=1).')
sessions_passing_clamp = [
    s for s, (t, a, n, tg) in alpha_session_summary.items()
    if ALPHA_LO <= a <= ALPHA_HI
]
print(f'  Sessions kept by alpha clamp [{ALPHA_LO}, {ALPHA_HI}]: '
      f'{len(sessions_passing_clamp)} -- {sessions_passing_clamp}')
for row in recal_session_summary:
    sess = row['session']
    raw_tcal = T_cal_true_by_session.get(sess)
    applied_vals = applied_alpha_by_session.get(sess, [])
    applied_med = float(np.median(applied_vals)) if applied_vals else float('nan')
    if applied_vals:
        applied_tag = (f'applied alpha (median over {len(applied_vals)} cells) '
                       f'= {applied_med:.3f} -> T_cal_eff={T_CAL_11 * applied_med:.1f} K')
    else:
        applied_tag = 'no cells'
    if raw_tcal is not None:
        raw_tag = (f', raw recal alpha={raw_tcal/T_CAL_11:.3f} '
                   f'(n_pts={row["n_alpha_points"]}, targets={",".join(row["targets"])})')
    else:
        raw_tag = ', no recal data in this session'
    print(f'  {sess}: {applied_tag}{raw_tag}')

Recal visits: 6 across 3 (session, target) pairs
  main/session_001 / recal_drift: 1 visits (1 with R, n_pairs>=2), T_sys_pipe=240.8 K, Y=1.3281, alt=49.1 deg
  main/session_002 / recal_drift: 4 visits (4 with R, n_pairs>=2), T_sys_pipe=207.0 K, Y=1.3817, alt=51.6 deg
  main/session_003 / recal_drift: 1 visits (1 with R, n_pairs>=2), T_sys_pipe=201.3 K, Y=1.3924, alt=52.9 deg
Wrote artifacts/T_B_ref_circumpolar.npz (1 target(s), anchor: main/session_001).

alpha points (session aggregation): 1 kept, 0 dropped outside [0.5, 2.0]
  alpha distribution: median=1.0912, p10=1.0912, p90=1.0912, range=[1.0912, 1.0912]
  alpha vs alt (per visit, all values): slope=+0.0197/deg, Pearson r=+0.418 (target: ~0)

Drift correction applied: 24 cells scaled, 0 unchanged (alpha=1).
  Sessions kept by alpha clamp [0.5, 2.0]: 1 -- ['main/session_002']
  main/session_002: applied alpha (median over 24 cells) = 1.091 -> T_cal_eff=86.2 K, raw recal alpha=1.091 (n_pts=4, targets=recal_drift)


/tmp/ipykernel_106660/3037911711.py:50: RuntimeWarning: Mean of empty slice
  P_on = np.nanmean([c['corr11'] for c in cals], axis=0)
/tmp/ipykernel_106660/3037911711.py:51: RuntimeWarning: Mean of empty slice
  P_off = np.nanmean([o['corr11'] for o in obss], axis=0)
/tmp/ipykernel_106660/3037911711.py:72: RuntimeWarning: Mean of empty slice
  I1 = np.nanmean([o['stokes_I'] for o in o1[:n_p]], axis=0)
/tmp/ipykernel_106660/3037911711.py:73: RuntimeWarning: Mean of empty slice
  I2 = np.nanmean([o['stokes_I'] for o in o2[:n_p]], axis=0)
/tmp/ipykernel_106660/3037911711.py:165: RuntimeWarning: Mean of empty slice
  R_shape = np.nanmean(R_stack, axis=0)


## 6. LSR correction, cross-session pair filter, and combine

Build per-pair R for each cell across all sessions, LSR-shift each pair onto a
common grid, reject outlier pairs via population MAD (cross-session), then
average the surviving pairs. Cells with fewer than `MIN_VIABLE_PAIRS` after
filtering are queued for reobservation.

In [9]:
print('Computing LSR corrections...')
cell_dr_groups = defaultdict(list)
for r in records:
    if r.get('gl') is None or r['noise_on']:
        continue
    cell_dr_groups[(r['session'], r['gl'], r['gb'])].append(r)

session_cell_vcorr = {}
for key, group in cell_dr_groups.items():
    r0 = group[0]
    mean_t = np.mean([r['time'] for r in group])
    session_cell_vcorr[key] = vlsr_correction(r0['ra'], r0['dec'], mean_t)

sci_vcorr = list(session_cell_vcorr.values())
mean_vcorr = np.mean(sci_vcorr) if sci_vcorr else 0.0
v_lsr_overlap = v_overlap + mean_vcorr
v_lsr_inc = v_lsr_overlap[::-1]

if sci_vcorr:
    print(f'v_corr range: {min(sci_vcorr):.2f} to {max(sci_vcorr):.2f} km/s '
          f'(mean {mean_vcorr:.2f})')
    print(f'Velocity (LSR): [{v_lsr_overlap[-1]:.0f}, {v_lsr_overlap[0]:.0f}] km/s')
else:
    print('No science dumps -- skipping LSR correction')


def _interp_to_lsr(spec_topo, v_topo_inc, v_lsr_inc_):
    """Interp a topocentric overlap spectrum onto the common LSR grid."""
    return np.interp(v_lsr_inc_, v_topo_inc[::-1],
                     spec_topo[::-1],
                     left=np.nan, right=np.nan)[::-1]


# Build per-pair (R_lsr, T_sys_lsr, T_B_lsr) for each (gl, gb), all on the
# common LSR grid. T_sys is now channel-dependent: T_B_lsr(v) = R_lsr(v) *
# T_sys_lsr(v) per pair, so spectral structure of T_sys (bandpass curvature,
# diode response) propagates correctly through the cross-session average.
cell_pairs = defaultdict(list)
n_pairs_no_tsys = 0

for (dr, gl, gb), dumps in obs_dumps_by_cell.items():
    d1 = [r for r in dumps if r['lo_mhz'] == lo1]
    d2 = [r for r in dumps if r['lo_mhz'] == lo2]
    n_p = min(len(d1), len(d2))
    if n_p == 0:
        continue

    tb_key = (dr, gl, gb)
    tb_entry = cell_results_TB.get(tb_key)
    if tb_entry is None or not np.isfinite(tb_entry['T_sys']):
        n_pairs_no_tsys += n_p
        continue

    T_sys_overlap = tb_entry['T_sys_overlap']  # topocentric, overlap region

    I1 = np.array([r['stokes_I'] for r in d1[:n_p]])
    I2 = np.array([r['stokes_I'] for r in d2[:n_p]])
    R_pairs = (I1 - I2) / I2  # (n_p, NFFT)
    R_pairs_ov = R_pairs[:, overlap_mask]

    v_corr = session_cell_vcorr.get((dr, gl, gb), 0.0)
    v_shifted = v_overlap + v_corr

    T_sys_lsr = _interp_to_lsr(T_sys_overlap, v_shifted, v_lsr_inc)

    for i in range(n_p):
        R_lsr = _interp_to_lsr(R_pairs_ov[i], v_shifted, v_lsr_inc)
        cell_pairs[(gl, gb)].append({
            'session': dr,
            'pair_idx': i,
            'R_lsr': R_lsr,
            'T_sys_lsr': T_sys_lsr,
            'T_B_lsr': R_lsr * T_sys_lsr,
        })

if n_pairs_no_tsys:
    print(f'Skipped {n_pairs_no_tsys} pairs with missing T_sys')

# Cross-session per-pair outlier filter on T_B (K).
n_pairs_total = sum(len(v) for v in cell_pairs.values())
viable_pairs_per_cell, flagged_pair_records = flag_outlier_pairs(
    cell_pairs,
    n_sigma=PAIR_NSIGMA,
    frac_thresh=PAIR_FRAC_THRESH,
    min_pairs=MIN_VIABLE_PAIRS,
    spectrum_key=PAIR_SPECTRUM_KEY,
)
print(f'Pair filter (on {PAIR_SPECTRUM_KEY}): flagged '
      f'{len(flagged_pair_records)}/{n_pairs_total} pairs '
      f'(NSIGMA={PAIR_NSIGMA}, frac>{PAIR_FRAC_THRESH:.0%})')

# Aggregate viable pairs into cell_combined. T_B carries each pair's
# channel-dependent T_sys; T_sys_nu is the average T_sys spectrum on the
# LSR grid; T_sys (scalar) is the band-median for QA / legacy reporting.
cell_combined = {}
cells_insufficient_pairs = []

for (gl, gb), pairs in viable_pairs_per_cell.items():
    TB_list = [p['T_B_lsr'] for p in pairs]
    R_list = [p['R_lsr'] for p in pairs]
    Tsys_lsr_list = [p['T_sys_lsr'] for p in pairs]

    n_viable = len(TB_list)
    if n_viable == 0:
        cells_insufficient_pairs.append({'l': gl, 'b': int(gb), 'n_viable': 0})
        continue

    T_sys_nu = np.nanmean(Tsys_lsr_list, axis=0)
    cell_combined[(gl, gb)] = {
        'T_B': np.nanmean(TB_list, axis=0),
        'R': np.nanmean(R_list, axis=0),
        'T_sys_nu': T_sys_nu,
        'T_sys': float(np.nanmedian(T_sys_nu)),
        'n_pairs': n_viable,
    }

    if n_viable < MIN_VIABLE_PAIRS:
        cells_insufficient_pairs.append({'l': gl, 'b': int(gb), 'n_viable': n_viable})

n_valid = sum(1 for v in cell_combined.values()
              if np.count_nonzero(np.isfinite(v['T_B'])) >= 8)
print(f'Combined (LSR): {len(cell_combined)} cells ({n_valid} with >=8 valid channels)')
print(f'Cells with <{MIN_VIABLE_PAIRS} viable pairs: {len(cells_insufficient_pairs)}')

# Sanity check: scalar-T_sys vs spectral-T_sys integrated W on (l=120, b=0)
# (the canonical lab-manual reference cell). Difference should be small (few %)
# if the spectral correction is well-behaved.
ref_key = (120.0, 0)
ref_alt_keys = [k for k in cell_combined if abs(k[0] - 120.0) < 0.5 and k[1] == 0]
if ref_alt_keys:
    rk = ref_alt_keys[0]
    cv = cell_combined[rk]
    W_spectral = float(np.nansum(cv['T_B']) * dv_kms)
    W_scalar = float(np.nansum(cv['R'] * cv['T_sys']) * dv_kms)
    if W_scalar != 0:
        delta = (W_spectral - W_scalar) / W_scalar
        print(f'Sanity (l={rk[0]:.2f}, b={rk[1]}): '
              f'W_spectral={W_spectral:.0f}, W_scalar={W_scalar:.0f} K*km/s, '
              f'delta={delta*100:+.2f}%')
else:
    print('Sanity-check cell (l~120, b=0) not present in cell_combined -- skipping.')

Computing LSR corrections...
v_corr range: -40.17 to -36.40 km/s (mean -38.37)
Velocity (LSR): [-206, 90] km/s
Pair filter (on T_B_lsr): flagged 1/96 pairs (NSIGMA=5.0, frac>15%)
Combined (LSR): 24 cells (24 with >=8 valid channels)
Cells with <3 viable pairs: 0
Sanity-check cell (l~120, b=0) not present in cell_combined -- skipping.


/home/ikaros/projects/ay-121/labs/04/utils/qa.py:555: RuntimeWarning: All-NaN slice encountered
  median = np.nanmedian(R_stack, axis=0)
/home/ikaros/projects/ay-121/labs/04/utils/qa.py:556: RuntimeWarning: All-NaN slice encountered
  mad = np.nanmedian(np.abs(R_stack - median), axis=0) * 1.4826
/tmp/ipykernel_106660/1142215317.py:108: RuntimeWarning: Mean of empty slice
  T_sys_nu = np.nanmean(Tsys_lsr_list, axis=0)
/tmp/ipykernel_106660/1142215317.py:110: RuntimeWarning: Mean of empty slice
  'T_B': np.nanmean(TB_list, axis=0),
/tmp/ipykernel_106660/1142215317.py:111: RuntimeWarning: Mean of empty slice
  'R': np.nanmean(R_list, axis=0),


# N. Neighbour-based QA

In [10]:
# Neighbor QA disabled.  Re-enable by uncommenting below and deleting `neighbor_cells = []`.
neighbor_cells = []
# # QA runs on T_B (K), so W is in K*km/s and noise_rms is in K. This makes the
# # neighbor comparison and W_sigma_floor physically meaningful and not biased
# # by cell-to-cell T_sys variation.
# cell_combined_qa = {
#     k: {'R_overlap': v['T_B'], 'n_pairs': v['n_pairs']}
#     for k, v in cell_combined.items()
# }
#
# cell_metrics = compute_cell_metrics(
#     cell_combined_qa, v_lsr_overlap, dv_kms,
#     min_valid_ch=METRIC_MIN_VALID_CH,
#     noise_v_max_kms=METRIC_NOISE_V_MAX_KMS,
#     signal_v_lo_kms=METRIC_SIGNAL_V_LO_KMS,
#     signal_v_hi_kms=METRIC_SIGNAL_V_HI_KMS,
#     smooth_kernel=METRIC_SMOOTH_KERNEL,
#     peak_min_sep_kms=METRIC_PEAK_MIN_SEP_KMS,
#     peak_prom_nsigma=METRIC_PEAK_PROM_NSIGMA,
#     min_noise_ch=METRIC_MIN_NOISE_CH,
# )
# neighbor_cells = neighbor_qa(
#     cell_metrics,
#     dv_kms=dv_kms,
#     hpbw_deg=HPBW_DEG,
#     neighbor_max_sep_deg=NEIGHBOR_MAX_SEP_DEG,
#     min_neighbors=MIN_NEIGHBORS,
#     w_z_thresh=W_Z_THRESH,
#     w_frac_thresh=W_FRAC_THRESH,
#     w_scale_floor=W_SCALE_FLOOR,
#     peak_v_z_thresh=PEAK_V_Z_THRESH,
#     peak_v_abs_thresh=PEAK_V_ABS_THRESH,
#     peak_v_min_sigma=PEAK_V_MIN_SIGMA,
#     peak_v_scale_floor=PEAK_V_SCALE_FLOOR,
#     bimodal_min_ratio=BIMODAL_MIN_RATIO,
# )
#
# neighbor_flagged = [c for c in neighbor_cells if c['W_flag'] or c['peak_v_flag']]
# neighbor_w_flag_count = sum(1 for c in neighbor_cells if c['W_flag'])
# neighbor_peak_flag_count = sum(1 for c in neighbor_cells if c['peak_v_flag'])
#
# print(f'Neighbor QA: {len(neighbor_cells)} cells analyzed (on T_B)')
# print(f'  Flags: W={neighbor_w_flag_count}, peak_v={neighbor_peak_flag_count}, '
#       f'any={len(neighbor_flagged)}')
#
# if neighbor_flagged:
#     print('  Most deviant cells:')
#
#     def severity(cell):
#         w_score = abs(cell['W_frac_resid']) if np.isfinite(cell['W_frac_resid']) else 0.0
#         v_score = abs(cell['peak_v_z']) if np.isfinite(cell['peak_v_z']) else 0.0
#         return max(w_score, v_score)
#
#     for cell in sorted(neighbor_flagged, key=severity, reverse=True)[:12]:
#         print(
#             f"    l={cell['gl']:6.2f} b={cell['gb']:3d} "
#             f"W={cell['W']:+.1f} K*km/s (frac={cell['W_frac_resid']:+.2f}, z={cell['W_z']:+.2f}) "
#             f"v_peak={cell['peak_v']:+.1f} km/s "
#             f"(dv={cell['peak_v_resid']:+.1f}, z={cell['peak_v_z']:+.2f}) "
#             f"n={cell['neighbor_count']}"
#         )


# N+. T_sys QA

Per-cell `T_sys` is the mean across viable pairs in `cell_combined`. Flag
cells whose `T_sys` lies more than `T_SYS_NSIGMA` robust standard
deviations (1.4826 * MAD) from the population median. Severe T_sys
excursions usually point to a bad cal-on/off block (noise diode partially
fired, scan boundary inside the cal pair, or unusually high ambient
temperature), and propagate silently into `T_B` since `T_B = R * T_sys`.

In [11]:
# T_sys QA disabled.  Re-enable by uncommenting below and deleting `tsys_flagged = []`.
tsys_flagged = []
# T_SYS_NSIGMA = 4.0
# T_SYS_ABS_LO = 80.0    # K, hard floor: any cell below this is suspect
# T_SYS_ABS_HI = 300.0   # K, hard ceiling
#
# # Build per-cell T_sys array from cell_combined
# tsys_cells = []
# for (gl, gb), v in cell_combined.items():
#     tsys = float(v.get('T_sys', np.nan))
#     if not np.isfinite(tsys):
#         continue
#     tsys_cells.append({'gl': gl, 'gb': gb, 'T_sys': tsys,
#                        'n_pairs': v.get('n_pairs', 0)})
#
# tsys_arr = np.array([c['T_sys'] for c in tsys_cells])
# tsys_med = float(np.median(tsys_arr))
# tsys_mad = float(np.median(np.abs(tsys_arr - tsys_med)))
# tsys_sigma = 1.4826 * tsys_mad if tsys_mad > 0 else np.nan
#
# for c in tsys_cells:
#     z = (c['T_sys'] - tsys_med) / tsys_sigma if np.isfinite(tsys_sigma) else 0.0
#     c['T_sys_z'] = z
#     c['T_sys_flag'] = (abs(z) > T_SYS_NSIGMA
#                       or c['T_sys'] < T_SYS_ABS_LO
#                       or c['T_sys'] > T_SYS_ABS_HI)
#
# tsys_flagged = [c for c in tsys_cells if c['T_sys_flag']]
#
# print(f'T_sys QA: {len(tsys_cells)} cells analyzed')
# print(f'  median = {tsys_med:.1f} K, robust sigma = {tsys_sigma:.1f} K')
# print(f'  threshold: |z| > {T_SYS_NSIGMA} OR T_sys outside '
#       f'[{T_SYS_ABS_LO:.0f}, {T_SYS_ABS_HI:.0f}] K')
# print(f'  Flagged: {len(tsys_flagged)} cells')
#
# if tsys_flagged:
#     print('  Most deviant cells:')
#     for c in sorted(tsys_flagged, key=lambda x: abs(x['T_sys_z']),
#                     reverse=True)[:12]:
#         print(f"    l={c['gl']:6.2f} b={c['gb']:+3d} "
#               f"T_sys={c['T_sys']:6.1f} K  z={c['T_sys_z']:+.2f}  "
#               f"n_pairs={c['n_pairs']}")
#


In [12]:
# Cells to reobserve: union of neighbor-QA flags, insufficient pairs, T_sys outliers.
reobs_map = {}  # (l, b) -> {'l', 'b', 'reason'}

for c in neighbor_cells:
    if not (c['W_flag'] or c['peak_v_flag']):
        continue
    flags = []
    if c['W_flag']:
        flags.append('W')
    if c['peak_v_flag']:
        flags.append('peak_v')
    key = (c['gl'], int(c['gb']))
    reobs_map[key] = {'l': c['gl'], 'b': int(c['gb']),
                      'reason': '+'.join(flags)}

for c in cells_insufficient_pairs:
    key = (c['l'], int(c['b']))
    tag = f"insufficient_pairs(n={c['n_viable']})"
    if key in reobs_map:
        reobs_map[key]['reason'] += '+' + tag
    else:
        reobs_map[key] = {'l': c['l'], 'b': int(c['b']), 'reason': tag}

for c in tsys_flagged:
    key = (c['gl'], int(c['gb']))
    tag = f"T_sys({c['T_sys']:.0f}K,z={c['T_sys_z']:+.1f})"
    if key in reobs_map:
        reobs_map[key]['reason'] += '+' + tag
    else:
        reobs_map[key] = {'l': c['gl'], 'b': int(c['gb']), 'reason': tag}

reobs = sorted(reobs_map.values(), key=lambda r: (r['b'], r['l']))

REOBSERVE_PATH.write_text(json.dumps(reobs, indent=2) + '\n')
print(f'Wrote {REOBSERVE_PATH} -- {len(reobs)} cells')
for r in reobs:
    print(f"  l={r['l']:7.2f} b={r['b']:+3d}  ({r['reason']})")


Wrote artifacts/main_reobserve.json -- 0 cells


In [ ]:
# Cells the rest of the pipeline does not trust.
qa_flagged_set = {(c['gl'], c['gb']) for c in neighbor_cells
                  if c['W_flag'] or c['peak_v_flag']}
# (T_sys exclusion disabled) qa_flagged_set |= {(c['gl'], c['gb']) for c in tsys_flagged}
insufficient_set = {(c['l'], c['b']) for c in cells_insufficient_pairs}
excluded_cells = qa_flagged_set | insufficient_set

# Build per-(session, cell) R from the cross-session-filtered viable pairs.
# (Spectra are on the common LSR grid, not topocentric.)
viable_per_session_cell = defaultdict(list)
for (gl, gb), pairs in viable_pairs_per_cell.items():
    if (gl, gb) in excluded_cells:
        continue
    for p in pairs:
        viable_per_session_cell[(p['session'], gl, gb)].append(p['R_lsr'])


def _safe_nanmean_stack(rs):
    """nanmean over axis 0 without a warning when some columns are all-NaN."""
    stack = np.array(rs)
    col_has_data = np.any(np.isfinite(stack), axis=0)
    mean = np.full(stack.shape[1], np.nan)
    if col_has_data.any():
        mean[col_has_data] = np.nanmean(stack[:, col_has_data], axis=0)
    return mean


with PdfPages('artifacts/spectra_per_session.pdf') as pdf:
    for dr in sessions:
        dr_spectra = {
            (l, b): _safe_nanmean_stack(rs)
            for (d, l, b), rs in viable_per_session_cell.items()
            if d == dr
        }
        if not dr_spectra:
            continue
        result = plot_spectra_grid(
            v_lsr_overlap, dr_spectra,
            ncols=5,
            color='C0',
            title=f'{dr} -- {len(dr_spectra)} pointings '
                  f'(post pair filter + QA, LSR frame)',
        )
        figs = result if isinstance(result, list) else [result]
        for f in figs:
            pdf.savefig(f)
            plt.close(f)

n_excluded = len(excluded_cells)
n_tsys = sum(1 for c in tsys_flagged)
print(f'Saved spectra_per_session.pdf '
      f'(excluded {n_excluded} cells: {len(qa_flagged_set)} QA '
      f'[incl. {n_tsys} T_sys] + {len(insufficient_set)} insufficient-pairs)')


## 7a. Integrated frequency-switched ratio map W_R(l, b)

Same integration, but on the dimensionless frequency-switched ratio R(v)
rather than the calibrated brightness temperature T_B. Useful as a
sanity check decoupled from the per-cell T_sys calibration: structure
appearing here but not in W(l, b) (or vice versa) points to gain or
T_sys issues rather than astrophysics.

In [ ]:
# QA-flagged cells to exclude (neighbor + T_sys outliers)
qa_flagged_set = {(c['gl'], c['gb']) for c in neighbor_cells
                  if c['W_flag'] or c['peak_v_flag']}
# (T_sys exclusion disabled) qa_flagged_set |= {(c['gl'], c['gb']) for c in tsys_flagged}

# Build arrays from cell_combined, excluding QA-flagged cells
cell_keys_clean = [k for k in cell_combined if k not in qa_flagged_set]
gl_arr = np.array([k[0] for k in cell_keys_clean])
gb_arr = np.array([k[1] for k in cell_keys_clean])
R_stack = np.array([cell_combined[k]['R'] for k in cell_keys_clean])

# Integrated frequency-switched ratio: W_R = sum(R) * dv
W_R = np.nansum(R_stack, axis=1) * dv_kms

valid_R = np.isfinite(W_R)

fig, ax = plot_survey_mollweide(
    gl_arr[valid_R], gb_arr[valid_R], W_R[valid_R],
    center_l=MOLL_CENTER_L,
    cbar_label=r'$W_R = \Sigma\, R \cdot \Delta v$ [km s$^{-1}$]',
    cmap='inferno',
    marker_size=(SS_FINE + SS_MICRO) / 2,
    title=f'Integrated HI ratio R -- {valid_R.sum()} cells ({len(qa_flagged_set)} QA-excluded)',
)
plt.show()


## 7. Integrated intensity map W(l, b)

In [ ]:
# QA-flagged cells to exclude (neighbor + T_sys outliers)
qa_flagged_set = {(c['gl'], c['gb']) for c in neighbor_cells
                  if c['W_flag'] or c['peak_v_flag']}
# (T_sys exclusion disabled) qa_flagged_set |= {(c['gl'], c['gb']) for c in tsys_flagged}

# Build arrays from cell_combined, excluding QA-flagged cells
cell_keys_clean = [k for k in cell_combined if k not in qa_flagged_set]
gl_arr = np.array([k[0] for k in cell_keys_clean])
gb_arr = np.array([k[1] for k in cell_keys_clean])
TB_stack = np.array([cell_combined[k]['T_B'] for k in cell_keys_clean])

# Integrated intensity: W = sum(T_B) * dv
W = np.nansum(TB_stack, axis=1) * dv_kms

valid = np.isfinite(W)

fig, ax = plot_survey_mollweide(
    gl_arr[valid], gb_arr[valid], W[valid],
    center_l=MOLL_CENTER_L,
    cbar_label=r'$W = \Sigma\, T_B \cdot \Delta v$ [K km s$^{-1}$]',
    cmap='inferno',
    marker_size=(SS_FINE + SS_MICRO) / 2,
    title=f'Integrated HI intensity -- {valid.sum()} cells ({len(qa_flagged_set)} QA-excluded)',
)
plt.show()


In [ ]:
# Export state for scan_load_lv.ipynb / scan_load_diagnostics.ipynb.
import pickle

STATE_PATH = Path('artifacts/scan_load_state.pkl')
tsys_by_session_cell = {
    key: float(v['T_sys'])
    for key, v in cell_results_TB.items()
    if np.isfinite(v.get('T_sys', np.nan))
}
tsys_nu_by_session_cell = {
    key: np.asarray(v['T_sys_nu'], dtype=float)
    for key, v in cell_results_TB.items()
    if np.isfinite(v.get('T_sys', np.nan))
}
tsys_overlap_by_session_cell = {
    key: np.asarray(v['T_sys_overlap'], dtype=float)
    for key, v in cell_results_TB.items()
    if np.isfinite(v.get('T_sys', np.nan))
}
gain_by_session_cell = {
    key: float(v[CAL_GAIN_KEY + '_scalar'])
    for key, v in cell_results_TB.items()
    if np.isfinite(v.get(CAL_GAIN_KEY + '_scalar', np.nan))
}
gain_nu_by_session_cell = {
    key: np.asarray(v[CAL_GAIN_KEY], dtype=float)
    for key, v in cell_results_TB.items()
    if v.get(CAL_GAIN_KEY) is not None
}
# Per-cell drift correction factor that was applied above (1.0 if recal data
# was unavailable for that cell's session). Exported so the diagnostics
# notebook can plot the recal timeline alongside the survey cells.
drift_alpha_by_session_cell = {
    key: float(v.get('drift_alpha', 1.0))
    for key, v in cell_results_TB.items()
}
# Per-visit recal calibration time series. Keys: 'session||target_id'.
# `T_sys` is kept as an alias for `T_sys_pipeline` so the legacy
# diagnostics panel still works without changes.
recal_visits_export = {
    f'{sess}||{tgt}': [
        {
            't_mid': float(v['t_mid']),
            'T_sys': float(v['T_sys_pipeline']),
            'T_sys_pipeline': float(v['T_sys_pipeline']),
            'T_sys_true': float(v.get('T_sys_true', float('nan'))),
            'T_cal_true': float(v.get('T_cal_true', float('nan'))),
            'Y_scalar': float(v.get('Y_scalar', float('nan'))),
            'alpha_abs': float(v.get('alpha_abs', float('nan'))),
            'g_T_cal': float(v['g_T_cal']),
            'n_cal': int(v['n_cal']),
            'n_obs': int(v['n_obs']),
            'n_pairs': int(v.get('n_pairs', 0)),
        }
        for v in vs
    ]
    for (sess, tgt), vs in recal_visits.items()
}
state = {
    'cell_combined': cell_combined,
    'qa_flagged_set': qa_flagged_set,
    'v_lsr_overlap': v_lsr_overlap,
    'dv_kms': dv_kms,
    'tsys_by_session_cell': tsys_by_session_cell,
    'tsys_nu_by_session_cell': tsys_nu_by_session_cell,
    'tsys_overlap_by_session_cell': tsys_overlap_by_session_cell,
    'gain_by_session_cell': gain_by_session_cell,
    'gain_nu_by_session_cell': gain_nu_by_session_cell,
    'overlap_mask': overlap_mask,
    'sessions': sessions,
    'T_CAL_11': T_CAL_11,
    'drift_alpha_by_session_cell': drift_alpha_by_session_cell,
    'recal_visits': recal_visits_export,
    'obs_time_by_session_cell': obs_time_by_sc,
    # Per-session diagnostics for the absolute-Tcal drift correction.
    'T_sys_true_by_session': T_sys_true_by_session,
    'T_cal_true_by_session': T_cal_true_by_session,
    'ref_target_meta': ref_target_meta,
    'ref_tcal_anchor_session': REF_TCAL_ANCHOR_SESSION if USE_REF_TCAL else None,
}
with open(STATE_PATH, 'wb') as f:
    pickle.dump(state, f)
print(f'Wrote {STATE_PATH} ({STATE_PATH.stat().st_size/1e6:.2f} MB) -- '
      f'{len(cell_combined)} cells, {len(qa_flagged_set)} QA-flagged, '
      f'{len(tsys_by_session_cell)} (session,cell) T_sys entries '
      f'({len(tsys_nu_by_session_cell)} with T_sys(nu) spectra), '
      f'{sum(len(v) for v in recal_visits_export.values())} recal visits, '
      f'{len(T_cal_true_by_session)} sessions with T_cal_true')